# Minilink showcase

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/showcase_minilink.ipynb)

**Write the equations once. Simulate, analyze, control, plan, optimize, learn.**

Minilink is a Python framework for dynamical systems, control, optimization and learning.
A physical model is written once, as textbook equations, and becomes a `System`. Blocks
wired together are a `System` too. That one object is what every tool takes: classical and
modern control, planners, optimizers, reinforcement learning.

![The Minilink Backbone](https://raw.githubusercontent.com/alx87grd/minilink/main/docs/_static/pipeline_map.png)

This notebook runs that path end to end on a pendulum, in about a minute.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Ten lines

A plant from the catalog, a controller, and `@` to close the loop.

In [ ]:
import numpy as np

from minilink import Impedance, Pendulum

plant = Pendulum()
plant.x0 = np.array([2.0, 0.0])  # released at 2 rad

controller = Impedance(Kp=100.0, Kd=20.0)  # a spring and a damper on the joint
loop = controller @ plant

loop.compute_trajectory(tf=6.0)
loop.plot_trajectory()

In [ ]:
loop.animate()

## 2. Diagrams as code

A diagram is a Python expression: `@` closes a feedback loop, `>>` chains blocks, `+` puts
them in parallel, and `%` puts a block on a clock. `plot_diagram` draws what the
expression built.

In [ ]:
from minilink import Step

reference = Step(final_value=[1.0], step_time=1.0)
tracking = reference >> controller @ plant
tracking.plot_diagram()

The loop is itself a system: the same calls simulate it and plot it. The reference steps
to 1 rad; the spring settles a little short of it, where it balances gravity.

In [ ]:
plant.x0 = np.zeros(2)
tracking.compute_trajectory(tf=6.0)
tracking.plot_trajectory(signals=("x", "plant:u"))

`controller % 0.1` is the same law run by a computer at 10 Hz: the command is held
between two ticks, and the plant stays continuous.

In [ ]:
digital = reference >> (controller % 0.1) @ plant
digital.compute_trajectory(tf=6.0)
digital.plot_trajectory(signals=("x", "plant:u"))

## 3. Your own model

A model is its equations: subclass `DynamicSystem` and write $\dot x = f(x, u, t)$, and
$y = h(x, u, t)$ when the sensor does not read the whole state. Every tool in this
notebook then takes it like a catalog plant.

In [ ]:
from minilink import PID, DynamicSystem


class MassSpringDamper(DynamicSystem):
    def __init__(self):
        super().__init__(state_dim=2, input_dim=1, output_dim=1)

    def f(self, x, u, t=0, params=None):
        m, k, b = 1.0, 4.0, 0.3
        p, v = x
        a = (u[0] - b * v - k * p) / m
        return np.array([v, a])

    def h(self, x, u, t=0, params=None):
        return np.array([x[0]])  # the sensor measures the position only


mass = MassSpringDamper()
mass_loop = Step(final_value=[1.0], step_time=1.0) >> PID(Kp=10.0, Ki=4.0, Kd=2.0) @ mass
mass_loop.compute_trajectory(tf=10.0)
mass_loop.plot_trajectory()

## 4. One `f`, differentiated

`f` is a pure function, so its derivatives are exact: with respect to the state, the
input, or the physical parameters.

In [ ]:
x_up = np.array([np.pi, 0.0])  # the upright equilibrium

A = plant.jacobian("f", "x", x_up)  # the linear model at the upright: unstable
dfdp = plant.jacobian("f", "params", np.array([1.0, 0.0]))  # one sensitivity per parameter

print("A =", A.round(2).tolist(), " poles:", np.linalg.eigvals(A).round(2))
print("d(f)/d(mass) =", dfdp["m"].round(2), " d(f)/d(gravity) =", dfdp["gravity"].round(2))

The linear model at the upright is what `lqr` designs on. It returns a controller
block, and the pendulum balances.

In [ ]:
from minilink import lqr

balance = lqr(plant, Q=np.eye(2), R=np.eye(1), x_bar=x_up) @ plant
plant.x0 = np.array([np.pi - 0.6, 0.0])
balance.compute_trajectory(tf=4.0)
balance.animate()

Compiled with JAX, the same equations run in batches: a thousand pendulums, each with
its own rod length, in one call.

In [ ]:
import matplotlib.pyplot as plt

compiled = plant.compile(backend="jax")

lengths = np.linspace(0.5, 2.0, 1000)
starts = np.tile([1.0, 0.0], (1000, 1))
swings = compiled.rollout_batch(starts, n_steps=1000, dt=0.005, params=dict(plant.params, l=lengths))

t = 0.005 * np.arange(1001)
for k in range(0, 1000, 200):
    plt.plot(t, swings[k, :, 0], label=f"l = {lengths[k]:.2f} m")
plt.xlabel("t [s]")
plt.ylabel("theta [rad]")
plt.legend()
plt.show()

## 5. One problem, every planner

A `PlanningProblem` states a task and nothing about how to solve it: the system, a cost,
the allowed states and inputs, the start and the goal. Here: swing the pendulum up with a
motor too weak to lift it directly.

In [ ]:
from minilink import BallSet, PlanningProblem, QuadraticCost

plant.state.lower = np.array([-2.0 * np.pi, -8.0])
plant.state.upper = np.array([2.0 * np.pi, 8.0])

problem = PlanningProblem(
    plant,
    x0=np.zeros(2),
    Xf=BallSet(x_up, 0.2),
    cost=QuadraticCost.from_system(plant, Q=np.eye(2), R=np.eye(1), x_bar=x_up),
    X=plant.state.box,
    U=plant.input_box,
    horizon=4.0,
)

The same problem goes to three planners that share nothing but it. **Value iteration**
solves the Bellman equation on a grid and returns the cost-to-go from every state.

In [ ]:
from minilink import DynamicProgrammingPlanner

vi = DynamicProgrammingPlanner(
    problem.replace(horizon=None),  # the converged law: no horizon
    x_grid=(101, 101),
    u_grid=(11,),
    dt=0.05,
    tol=0.5,
    out_of_bound_cost=500.0,
)
vi.solve()
vi.plot_cost_to_go(jmax=500.0)

**RRT** grows a tree of motions until a branch reaches the goal.

In [ ]:
from minilink import RRTPlanner

rrt = RRTPlanner(problem, seed=0, max_iterations=5000)
rrt_solution = rrt.solve()
rrt.plot_tree(x_axis=0, y_axis=1);

**Trajectory optimization** transcribes the problem into a nonlinear program and solves
it. Each planner's answer holds a controller that is a `System`: value iteration's
closes the loop with the same `@`. The three answers on one phase plane:

In [ ]:
from minilink import TrajectoryOptimizationPlanner

trajopt_solution = TrajectoryOptimizationPlanner(problem, n_steps=40).solve()

plant.x0 = np.zeros(2)
vi_trajectory = (vi.controller @ plant).compute_trajectory(tf=10.0)

phase = plant.plot_phase_plane(traj=trajopt_solution.trajectory, show=False)
phase.axes.plot(*vi_trajectory.x, "C0--", label="value iteration")
phase.axes.plot(*rrt_solution.trajectory.x, "C2", label="RRT")
phase.axes.legend(loc="lower right")
plt.show()

A plan computed once is open loop. `ModelPredictiveController` solves it again at every
tick from the measured state: a planner on a clock is a controller.

In [ ]:
from minilink.control import ModelPredictiveController

horizon_cost = QuadraticCost.from_system(
    plant, x_bar=x_up, Q=np.eye(2), R=0.1 * np.eye(1), S=10.0 * np.eye(2)
)
horizon_problem = PlanningProblem(
    plant, x0=np.zeros(2), cost=horizon_cost, U=plant.input_box, horizon=3.0
)
horizon_planner = TrajectoryOptimizationPlanner(
    horizon_problem, n_steps=30, backend="jax", method="slsqp", options={"maxiter": 100}
)

mpc_loop = (ModelPredictiveController(horizon_planner, delay=True) % 0.1) @ plant
mpc_trajectory = mpc_loop.compute_trajectory(tf=6.0)
plant.animate(mpc_trajectory.of("plant"))

## 6. One optimization boundary

Every optimization in minilink, a trajectory, a set of gains, a network's weights, is
lowered to one object, a `MathematicalProgram`

$$\min_z J(z) \quad \text{s.t.} \quad g(z) \le 0, \quad h(z) = 0,$$

and any `Optimizer` solves any program. Here the program is the cost of a closed-loop
step response as a function of the controller's two gains, with its exact gradient
through the whole simulation.

In [ ]:
from minilink import Optimizer

step_cost = QuadraticCost.from_system(
    plant, Q=np.diag([1.0, 0.0]), R=0.01 * np.eye(1), x_bar=np.array([1.0, 0.0])
)
step_task = PlanningProblem(plant, x0=np.zeros(2), cost=step_cost, horizon=5.0)

evaluator = step_task.compile(Impedance(Kp=1.0, Kd=1.0), backend="jax")
start = evaluator.params
start["r"] = np.array([1.0])  # the reference: 1 rad

gains = ["controller.Kp", "controller.Kd"]
program = evaluator.program(gains, params=start)

for method in ("l-bfgs", "slsqp"):
    best = Optimizer(program, method=method).solve()
    print(f"{method:7s} Kp, Kd = {best.z.round(2)}  J* = {best.J:.3f}")

In [ ]:
tuned = evaluator.unpack(best.z, gains, params=start)
evaluator.loop.plot_trajectory(evaluator.trajectory(tuned))

## 7. Learn a law

Reinforcement learning is one more planner. It takes the same kind of problem, with a
start drawn at random, and returns a neural network that is, again, a controller block.

In [ ]:
import jax.numpy as jnp

from minilink import CostFunction, ReinforcementLearningPlanner, Uniform


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        return 1.0 + jnp.cos(x[0]) + 0.01 * x[1] ** 2 + 0.01 * u[0] ** 2

    def h(self, x, t=0.0, params=None):
        return 0.0


random_start = PlanningProblem(
    plant,
    cost=SwingUpCost(),
    horizon=np.inf,
    x0=Uniform([-np.pi, -1.0], [np.pi, 1.0]),
    U=plant.input_box,
)
learner = ReinforcementLearningPlanner(
    random_start,
    dt=0.05,
    features=lambda x: jnp.array([jnp.cos(x[0]), jnp.sin(x[0]), 0.1 * x[1]]),
    hidden=(32, 32),
    n_rollouts=64,
    n_steps_per_update=32,
    batch_size=256,
    learning_rate=3e-3,
    gamma=0.97,
)
learner.solve(n_timesteps=120_000)
learner.plot_learning_curve();

In [ ]:
neural_loop = learner.controller @ plant

plant.x0 = np.array([0.05, 0.0])
neural_loop.compute_trajectory(tf=8.0, dt=0.01)
neural_loop.animate()

One judge for every answer: `evaluate` scores a controller on the problem's own random
starts. The learned law against the LQR of section 4, on the same 100 draws; the LQR
ignores the motor's limit, and the count of feasible runs shows it.

In [ ]:
from minilink import evaluate

runs = dict(dt=0.05, n_trials=100, episode_length=10.0, key=1)

print(evaluate(random_start, learner.controller, **runs))  # the neural law
print(evaluate(random_start, balance.subsystems["controller"], **runs))  # the LQR

## 8. The same lines on a robot

Nothing above is special to the pendulum. A six-axis arm from the catalog, with its full
rigid-body dynamics, is the same kind of object.

In [ ]:
from minilink import JointImpedance, UR5Manipulator

arm = UR5Manipulator()
q_start = np.array([0.0, -np.pi / 2, 0.0, -np.pi / 2, 0.0, 0.0])
q_goal = np.array([1.35, -1.2, 1.55, -1.4, 0.35, 0.25])
arm.x0 = arm.q2x(q_start, np.zeros(6))

arm_law = JointImpedance(
    arm,
    gravity_comp=True,
    Kp=[10.0, 60.0, 20.0, 0.5, 0.1, 0.003],
    Kd=[3.0, 25.0, 8.0, 0.2, 0.05, 0.003],
)
arm_loop = Step(initial_value=q_start, final_value=q_goal, step_time=1.0) >> arm_law @ arm
arm_loop.compute_trajectory(tf=6.0, n_samples=180, backend="jax")
arm_loop.animate(renderer="meshcat", is_3d=True)

On your own machine, `plant.game()` puts the keyboard on a system's input in real time.

## Where to go next

- [`basics/`](basics/01_minilink.ipynb): model, connect, simulate, design a controller.
- [`advanced/`](advanced/01_discrete_and_hybrid.ipynb): sampled loops, optimal control, MPC, reinforcement learning, differentiable loops.
- [`core/`](core/01_systems.ipynb): the objects behind the tools, one by one.